# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR<sup>2</sup> dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nAvailable record sets:\n{[r.id for r in metadata.record_sets]}")

## 2. Data Overview
Review available record sets and their fields by their `@id` attribute.

In [ ]:
# List available record sets and their fields
for record_set in metadata.record_sets:
    print(f"Record set '@id': {record_set.id}, name: {record_set.name if hasattr(record_set, 'name') else ''}")
    if hasattr(record_set, 'fields'):
        print("  Fields (@id):")
        for field in record_set.fields:
            print(f"    - {field.id} ({field.name if hasattr(field, 'name') else ''})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references to record sets, fields, and columns are made exclusively via their `@id`s as per the Croissant schema.

In [ ]:
# Load all record set IDs for extraction
record_set_ids = [r.id for r in metadata.record_sets]
print("Record set IDs:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Only create DataFrame if there is data
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}.")

# Preview columns for the first non-empty record set
main_rs_id = next(iter(dataframes.keys()))
print(f"\nColumns in '{main_rs_id}':")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes, referencing only by `@id`. Typical operations include removing outliers, transforming distributions, or aggregation.

In [ ]:
# For the example, we try common proteomics quantitative fields.
# Adjust these IDs based on actual schema output from previous step if different.
df = dataframes[main_rs_id]

# Guess a numeric field
candidate_numeric_fields = [col for col in df.columns if df[col].dtype == 'float64' or df[col].dtype == 'int64']
print("Numeric field candidates:", candidate_numeric_fields)

if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]  # Example: '@id' for e.g. 'Coverage(%)' or 'MW'
    print(f"Using numeric field '@id': {numeric_field_id}")
else:
    numeric_field_id = None

# Proceed only if we have a numeric field
if numeric_field_id:
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try to group by a categorical field if present
    candidate_group_fields = [col for col in df.columns if (df[col].dtype == 'object' or df[col].dtype.name == 'category') and col != numeric_field_id]
    if candidate_group_fields:
        group_field_id = candidate_group_fields[0]
        print(f"\nGrouping by field '@id': {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"Grouped mean of {numeric_field_id} by {group_field_id} (top 5):")
        print(grouped_df.head())
    else:
        print("No categorical field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatter with another numeric field if available
    if len(candidate_numeric_fields) > 1:
        secondary_numeric_id = candidate_numeric_fields[1]
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[numeric_field_id], y=df[secondary_numeric_id])
        plt.xlabel(numeric_field_id)
        plt.ylabel(secondary_numeric_id)
        plt.title(f"Scatter plot of {numeric_field_id} vs {secondary_numeric_id}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we loaded a mass spectrometry dataset about extracellular vesicle proteins using the `mlcroissant` library, explored its record sets and fields by `@id`, extracted main data into DataFrames, filtered and normalized quantitative values, performed group-wise analysis, and visualized data distributions. Use this scaffold for deeper analysis and tailored research questions.